# Feed-forward Neural Network for Tabular Regression

10 float input features -> 1 continuous target, trained in PyTorch and evaluated with `torch.nn.MSELoss()`.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 1. Load data

`X_train.csv` / `X_test.csv`: 700 x 10 features, no header. `y_train.csv` / `Y_test.csv`: 700 x 1 target, no header. Features and target are already standardized, so no further scaling is applied.

In [ ]:
X_train_full = pd.read_csv('X_train.csv', header=None).values.astype(np.float32)
y_train_full = pd.read_csv('y_train.csv', header=None).values.astype(np.float32).reshape(-1, 1)
X_test = pd.read_csv('X_test.csv', header=None).values.astype(np.float32)
y_test = pd.read_csv('Y_test.csv', header=None).values.astype(np.float32).reshape(-1, 1)

print('X_train_full:', X_train_full.shape)
print('y_train_full:', y_train_full.shape)
print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

## 2. Train / validation split

A fixed-seed random 80/20 split of the training data (560 train / 140 validation), made once before training starts.

In [ ]:
n = X_train_full.shape[0]
rng = np.random.RandomState(SEED)
perm = rng.permutation(n)

n_val = int(round(0.2 * n))
val_idx = perm[:n_val]
train_idx = perm[n_val:]

X_train, y_train = X_train_full[train_idx], y_train_full[train_idx]
X_val, y_val = X_train_full[val_idx], y_train_full[val_idx]

print('train:', X_train.shape, 'val:', X_val.shape)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_X_t = torch.from_numpy(X_val).to(device)
val_y_t = torch.from_numpy(y_val).to(device)
test_X_t = torch.from_numpy(X_test).to(device)
test_y_t = torch.from_numpy(y_test).to(device)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

## 3. Model

Linear(10->64) + ReLU + Dropout(0.1) -> Linear(64->32) + ReLU + Dropout(0.1) -> Linear(32->16) + ReLU -> Linear(16->1). 4 Linear layers total, well under the 20-layer limit.

In [ ]:
class RegressionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(10, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)

model = RegressionNet().to(device)
model

## 4. Training

Adam (lr=1e-3, weight_decay=1e-4), `MSELoss`, up to 500 epochs, early stopping with patience=30 on validation MSE. The best-epoch weights are snapshotted into `best_model_state`.

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

MAX_EPOCHS = 500
PATIENCE = 30

train_losses = []
val_losses = []

best_val_loss = float('inf')
best_epoch = -1
best_model_state = None
epochs_without_improvement = 0

for epoch in range(MAX_EPOCHS):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
    train_mse = running_loss / len(train_ds)

    model.eval()
    with torch.no_grad():
        val_preds = model(val_X_t)
        val_mse = criterion(val_preds, val_y_t).item()

    train_losses.append(train_mse)
    val_losses.append(val_mse)

    if val_mse < best_val_loss:
        best_val_loss = val_mse
        best_epoch = epoch
        best_model_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d} | train MSE: {train_mse:.4f} | val MSE: {val_mse:.4f} | best val MSE: {best_val_loss:.4f} (epoch {best_epoch+1})')

    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs).')
        break

print(f'\nBest epoch: {best_epoch+1}, best validation MSE: {best_val_loss:.4f}')

## 5. Training curve

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_losses) + 1), train_losses, label='Train MSE')
plt.plot(range(1, len(val_losses) + 1), val_losses, label='Validation MSE')
plt.axvline(best_epoch + 1, color='gray', linestyle='--', label=f'Best epoch ({best_epoch+1})')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Training vs Validation MSE')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Reload best model and evaluate on the test set

In [ ]:
model.load_state_dict(best_model_state)
model.eval()

with torch.no_grad():
    test_preds = model(test_X_t)
    test_mse = criterion(test_preds, test_y_t).item()

print(f'Test MSE (best model, epoch {best_epoch+1}): {test_mse:.4f}')